# EU27 Climate Clustering — Optimal *k* Analysis

**Goal:** Determine the best number of clusters for country segmentation before EDA and time-series forecasting.

**Pipeline position:** `EDA → Clustering (this notebook) → Time Series Forecast`

**Input:** `EU27_sectors_scaled.csv` (StandardScaler-normalised, Country × Features)  
**Outputs:** `cluster_k_analysis.png`, `EU27_cluster_labels.csv`


In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from scipy.cluster.hierarchy import dendrogram, linkage
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# ── CONFIG — edit these ────────────────────────────────────────────────────────
DATA_PATH    = "EU27_sectors_scaled.csv"   # update path if needed
K_RANGE      = range(2, 9)
FINAL_K      = 4                           # swap to 3 to compare
RANDOM_STATE = 42
OUTPUT_IMG   = "cluster_k_analysis.png"
# ──────────────────────────────────────────────────────────────────────────────

print("Libraries loaded ✓")


## 1. Load Data

In [ ]:
df = pd.read_csv(DATA_PATH)
countries = df["Country"].values
codes     = df["Country_Code"].values
X         = df.drop(["Country", "Country_Code"], axis=1).values

print(f"Shape: {X.shape[0]} countries × {X.shape[1]} features")
print(f"Features: {list(df.drop(['Country','Country_Code'], axis=1).columns)}")
df.head()


## 2. Validation Metrics

Four metrics across k = 2–8:

| Metric | Direction | What it measures |
|--------|-----------|-----------------|
| **Inertia** | ↓ lower is better | Within-cluster sum of squared distances (elbow) |
| **Silhouette** | ↑ higher is better | How well each point fits its cluster vs neighbours |
| **Davies-Bouldin** | ↓ lower is better | Average ratio of within-cluster scatter to between-cluster distance |
| **Calinski-Harabasz** | ↑ higher is better | Ratio of between-cluster to within-cluster dispersion |


In [ ]:
inertia, sil, db, ch = [], [], [], []

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
    labels = km.fit_predict(X)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(X, labels))
    db.append(davies_bouldin_score(X, labels))
    ch.append(calinski_harabasz_score(X, labels))

results = pd.DataFrame({
    "k":                list(K_RANGE),
    "Inertia":          inertia,
    "Silhouette ↑":     sil,
    "Davies-Bouldin ↓": db,
    "Calinski-Harabasz ↑": ch,
})
results.set_index("k", inplace=True)
results.style.highlight_min(subset=["Inertia", "Davies-Bouldin ↓"], color="#ffe0e0") \
             .highlight_max(subset=["Silhouette ↑", "Calinski-Harabasz ↑"], color="#e0ffe0") \
             .format(precision=3)


## 3. PCA Projection (2D)

In [ ]:
pca   = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X)
var_exp = pca.explained_variance_ratio_ * 100

print(f"PC1: {var_exp[0]:.1f}%  |  PC2: {var_exp[1]:.1f}%  |  Total: {sum(var_exp):.1f}%")


## 4. Ward Hierarchical Linkage

In [ ]:
Z = linkage(X, method="ward")
print("Linkage matrix computed ✓")


## 5. Diagnostic Plot

In [ ]:
DARK_BG  = "#0f1117"
PANEL_BG = "#1a1d2e"
COLORS   = ["#00d4ff", "#ff6b6b", "#ffd700", "#7fff7f", "#ff9f43", "#a29bfe", "#fd79a8"]
k_list   = list(K_RANGE)

fig = plt.figure(figsize=(18, 14))
fig.patch.set_facecolor(DARK_BG)

# — Elbow ——————————————————————————————————————————————
ax1 = fig.add_subplot(2, 3, 1)
ax1.set_facecolor(PANEL_BG)
inertia_norm = np.array(inertia)
inertia_norm = (inertia_norm - inertia_norm.min()) / (inertia_norm.max() - inertia_norm.min())
ax1.plot(k_list, inertia_norm, color="#00d4ff", linewidth=2.5, marker="o",
         markersize=7, markerfacecolor="white")
for k_val in [3, 4, 5]:
    ax1.plot(k_val, inertia_norm[k_val - 2], "o", markersize=12,
             color="#ffd700", zorder=5, alpha=0.7)
ax1.axvline(x=4, color="#ffd700", linestyle="--", alpha=0.6, linewidth=1.5)
ax1.set_title("Elbow (Inertia)", color="white", fontsize=12, fontweight="bold", pad=10)
ax1.set_xlabel("k", color="#aaa"); ax1.set_ylabel("Normalised Inertia", color="#aaa")
ax1.tick_params(colors="#aaa"); ax1.spines[:].set_edgecolor("#333")

# — Silhouette ————————————————————————————————————————
ax2 = fig.add_subplot(2, 3, 2)
ax2.set_facecolor(PANEL_BG)
bar_colors = ["#ffd700" if k in [3, 4, 5] else "#00d4ff" for k in k_list]
ax2.bar(k_list, sil, color=bar_colors, alpha=0.85, edgecolor="none")
best_sil_k = k_list[sil.index(max(sil))]
ax2.axvline(x=best_sil_k, color="#ff6b6b", linestyle="--", linewidth=1.5, alpha=0.8)
for i, v in enumerate(sil):
    ax2.text(k_list[i], v + 0.005, f"{v:.2f}", ha="center", color="white", fontsize=9)
ax2.set_title("Silhouette Score ↑", color="white", fontsize=12, fontweight="bold", pad=10)
ax2.set_xlabel("k", color="#aaa"); ax2.set_ylabel("Score", color="#aaa")
ax2.tick_params(colors="#aaa"); ax2.spines[:].set_edgecolor("#333")

# — Davies-Bouldin ————————————————————————————————————
ax3 = fig.add_subplot(2, 3, 3)
ax3.set_facecolor(PANEL_BG)
ax3.plot(k_list, db, color="#ff9f43", linewidth=2.5, marker="s", markersize=7)
best_db_k = k_list[db.index(min(db))]
ax3.plot(best_db_k, db[best_db_k - 2], "*", markersize=15, color="#ffd700", zorder=5)
ax3.set_title("Davies-Bouldin ↓", color="white", fontsize=12, fontweight="bold", pad=10)
ax3.set_xlabel("k", color="#aaa"); ax3.set_ylabel("Score", color="#aaa")
ax3.tick_params(colors="#aaa"); ax3.spines[:].set_edgecolor("#333")

# — Dendrogram ————————————————————————————————————————
ax4 = fig.add_subplot(2, 3, 4)
ax4.set_facecolor(PANEL_BG)
dendrogram(Z, labels=codes, ax=ax4, color_threshold=5.5,
           above_threshold_color="#555", leaf_font_size=8)
ax4.axhline(y=5.5, color="#ffd700", linestyle="--", alpha=0.7,
            linewidth=1.5, label="4-cluster cut")
ax4.axhline(y=7.5, color="#ff6b6b", linestyle="--", alpha=0.7,
            linewidth=1.5, label="3-cluster cut")
ax4.legend(fontsize=8, facecolor=PANEL_BG, labelcolor="white")
ax4.set_title("Ward Dendrogram", color="white", fontsize=12, fontweight="bold", pad=10)
ax4.tick_params(colors="#aaa"); ax4.spines[:].set_edgecolor("#333")

# — PCA scatter k=3 and k=4 ———————————————————————————
for idx, k in enumerate([3, 4]):
    ax = fig.add_subplot(2, 3, 5 + idx)
    ax.set_facecolor(PANEL_BG)
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
    labels = km.fit_predict(X)
    for cl in range(k):
        mask = labels == cl
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
                   c=COLORS[cl], s=80, alpha=0.9,
                   edgecolors="white", linewidth=0.5)
        for i, code in enumerate(codes):
            if mask[i]:
                ax.annotate(code, (X_pca[i, 0], X_pca[i, 1]),
                            fontsize=7, color="white", alpha=0.85,
                            xytext=(4, 4), textcoords="offset points")
    sil_k = silhouette_score(X, labels)
    ax.text(0.02, 0.97, f"Sil: {sil_k:.3f}", transform=ax.transAxes,
            color="#ffd700", fontsize=9, va="top")
    ax.set_title(f"PCA — k={k}  ({sum(var_exp):.0f}% var)",
                 color="white", fontsize=12, fontweight="bold", pad=10)
    ax.set_xlabel(f"PC1 ({var_exp[0]:.1f}%)", color="#aaa")
    ax.set_ylabel(f"PC2 ({var_exp[1]:.1f}%)", color="#aaa")
    ax.tick_params(colors="#aaa"); ax.spines[:].set_edgecolor("#333")

plt.suptitle("EU27 Climate Clustering — Optimal k Analysis",
             color="white", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_IMG, dpi=160, bbox_inches="tight", facecolor=DARK_BG)
plt.show()
print(f"Figure saved → {OUTPUT_IMG}")


## 6. Final Cluster Assignment

In [ ]:
km_final = KMeans(n_clusters=FINAL_K, random_state=RANDOM_STATE, n_init=20)
final_labels = km_final.fit_predict(X)

print(f"Final clustering  (k={FINAL_K})")
print("=" * 50)
for c in range(FINAL_K):
    members = [countries[i] for i in range(len(countries)) if final_labels[i] == c]
    print(f"\nCluster {c+1} ({len(members)} countries):")
    print("  " + ", ".join(members))


## 7. Cluster Feature Profiles

Mean standardised feature values per cluster — useful for interpreting what drives each group.

In [ ]:
feature_cols = [c for c in df.columns if c not in ["Country", "Country_Code"]]
df_profile = df[feature_cols].copy()
df_profile["Cluster"] = final_labels + 1

profile = df_profile.groupby("Cluster").mean().T
profile.columns = [f"Cluster {c}" for c in profile.columns]
profile.style.background_gradient(cmap="RdYlGn", axis=1).format(precision=2)


## 8. Export Labels

In [ ]:
df_out = df[["Country", "Country_Code"]].copy()
df_out[f"Cluster_k{FINAL_K}"] = final_labels + 1   # 1-indexed

df_out.to_csv("EU27_cluster_labels.csv", index=False)
print("Saved → EU27_cluster_labels.csv")
df_out
